**Análise de Incêndios Florestais no Brasil (1997 - 2017)**

Os incêndios florestais representam um desafio crítico para a preservação dos biomas brasileiros, em especial a Floresta Amazônica, a maior floresta tropical do planeta.

**Objetivo do Projeto:**
Analisar a série histórica de queimadas no Brasil para identificar padrões de sazonalidade, evolução anual e mapear os estados mais afetados. Compreender a frequência e a distribuição dessas ocorrências é um passo fundamental para auxiliar no planejamento de medidas preventivas e na alocação estratégica de recursos.

**Base de Dados:** [Link do dataset original](https://drive.google.com/file/d/16PCjsLZuxmvxa0LlnTzCELbLVQhO37vN/view?usp=sharing)

> Análise exploratória dos registros de incêndios florestais no Brasil.


In [ ]:
# Bibliotecas para manipulação e visualização de dados
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Configuração de estilo padrão para os gráficos
sns.set_theme(style="whitegrid")


In [ ]:
# Lendo os dados diretamente do Google Drive
ID_ARQUIVO = "16PCjsLZuxmvxa0LlnTzCELbLVQhO37vN"
url = f"https://drive.google.com/uc?export=download&id={ID_ARQUIVO}"

Base_Dados = pd.read_csv(url, encoding="latin-1")

# Visualizando os primeiros registros
Base_Dados.head()


In [ ]:
# Verificando a quantidade de valores nulos por coluna
Base_Dados.isnull().sum()

In [ ]:
# Verificando valores nulos
nulos = Base_Dados.isnull().sum()

display(nulos)

if nulos.sum() == 0:
    print("\n✅ A base não possui valores nulos.")

In [ ]:
# Estatísticas descritivas das variáveis numéricas
Base_Dados.describe()

In [ ]:
# Informações gerais da base de dados
Base_Dados.info()

In [ ]:
# Quantidade de valores únicos por coluna
Base_Dados.nunique()

In [ ]:
# Total de incêndios por ano
Analise = (
    Base_Dados
    .groupby("year", as_index=False)["number"]
    .sum()
)

# Visualização da evolução anual
plt.figure(figsize=(14, 5))
plt.style.use("ggplot")

sns.lineplot(
    data=Analise,
    x="year",
    y="number",
    marker="o",
    linewidth=2
)

plt.title("Total de incêndios no Brasil por ano", fontsize=14)
plt.xlabel("Ano")
plt.ylabel("Número de incêndios")
plt.show()

In [ ]:
# Incêndios por ano e mês
Analise_02 = (
    Base_Dados
    .groupby(["year", "month"], as_index=False)["number"]
    .sum()
)

# Ordem cronológica dos meses
ordem_meses = [
    "Janeiro", "Fevereiro", "Março", "Abril",
    "Maio", "Junho", "Julho", "Agosto",
    "Setembro", "Outubro", "Novembro", "Dezembro"
]

# Distribuição mensal dos incêndios
plt.figure(figsize=(14, 5))

sns.boxplot(
    data=Analise_02,
    x="month",
    y="number",
    order=ordem_meses
)

plt.title("Distribuição dos incêndios por mês", fontsize=14)
plt.xlabel("Mês")
plt.ylabel("Número de incêndios")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Total de incêndios por estado
Analise_03 = (
    Base_Dados
    .groupby("state", as_index=False)["number"]
    .sum()
    .sort_values("number", ascending=False)
)

# Visualização dos incêndios por estado
plt.figure(figsize=(10, 8))

sns.barplot(
    data=Analise_03,
    x="number",
    y="state"
)

plt.title("Total de incêndios por estado", fontsize=14)
plt.xlabel("Número de incêndios")
plt.ylabel("Estado")
plt.show()

In [ ]:
# Selecionando os 10 estados com maior número de incêndios
Lista_TOP10 = (
    Analise_03
    .nlargest(10, "number")["state"]
    .tolist()
)

# Filtrando os dados dos estados selecionados
Analise_TOP10 = (
    Base_Dados[Base_Dados["state"].isin(Lista_TOP10)]
    .groupby(["year", "state"], as_index=False)["number"]
    .sum()
)

# Gráfico interativo
fig = px.line(
    Analise_TOP10,
    x="year",
    y="number",
    color="state",
    markers=True,
    title="Evolução anual dos incêndios nos 10 estados com maior ocorrência",
    labels={
        "year": "Ano",
        "number": "Número de incêndios",
        "state": "Estado"
    }
)

fig.update_layout(
    hovermode="x unified",
    legend_title_text="Estado"
)

fig.show()

In [ ]:
# Coordenadas aproximadas dos estados
coordenadas = {
    "Acre": (-8.77, -70.55),
    "Alagoas": (-9.71, -35.73),
    "Amapa": (1.41, -51.77),
    "Amazonas": (-3.07, -61.66),
    "Bahia": (-12.96, -38.51),
    "Ceara": (-3.71, -38.54),
    "Distrito Federal": (-15.83, -47.86),
    "Espirito Santo": (-19.19, -40.34),
    "Goias": (-16.64, -49.31),
    "Maranhao": (-2.55, -44.30),
    "Mato Grosso": (-12.64, -55.42),
    "Minas Gerais": (-18.10, -44.38),
    "Paraiba": (-7.06, -35.55),
    "Pará": (-5.53, -52.29),
    "Pernambuco": (-8.28, -35.07),
    "Piau": (-8.28, -43.68),
    "Rio": (-22.84, -43.15),
    "Rondonia": (-11.22, -62.80),
    "Roraima": (1.89, -61.22),
    "Santa Catarina": (-27.33, -49.44),
    "Sao Paulo": (-23.55, -46.64),
    "Sergipe": (-10.90, -37.07),
    "Tocantins": (-10.25, -48.25)
}

# Criando DataFrame com as coordenadas
df_coordenadas = pd.DataFrame(
    [
        {
            "state": estado,
            "Latitude": latitude,
            "Longitude": longitude
        }
        for estado, (latitude, longitude) in coordenadas.items()
    ]
)

# Unindo as coordenadas aos totais de incêndios por estado
Analise_Geografica = Analise_03.merge(
    df_coordenadas,
    on="state",
    how="left"
)

# Renomeando para exibição
Analise_Geografica = Analise_Geografica.rename(
    columns={
        "state": "Estado",
        "number": "Incêndios"
    }
)

Analise_Geografica.head()


In [ ]:
# Mapa de calor geográfico dos incêndios
fig = px.density_map(
    Analise_Geografica,
    lat="Latitude",
    lon="Longitude",
    z="Incêndios",
    radius=30,
    center={
        "lat": -14.0,
        "lon": -52.0
    },
    zoom=3,
    map_style="open-street-map",
    hover_name="Estado",
    title="Distribuição geográfica dos incêndios no Brasil"
)

fig.update_layout(
    margin={"r": 0, "t": 50, "l": 0, "b": 0}
)

fig.show()